# Validation-Based Model Selection and Classifier Comparison



## 1. Functions (identical to the original notebooks)

In [1]:
import argparse
import os
import time
import warnings

warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
from scipy.stats import friedmanchisquare, wilcoxon
from sklearn.ensemble import AdaBoostClassifier, RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.svm import LinearSVC

import lightgbm as lgb
import xgboost as xgb

try:
    import tensorflow as tf
    from tensorflow import keras
    tf.get_logger().setLevel("ERROR")
    HAS_TF = True
except ImportError:
    HAS_TF = False

# ----------------------------- settings (identical to the notebook) -----------------------------
RANDOM_STATE = 0
N_COMPARISON_REPEATS = 20
N_CPU_THREADS = 2
CNN_EPOCHS = 100
ADABOOST_N_ESTIMATORS = 200
RF_N_ESTIMATORS = 500

np.random.seed(RANDOM_STATE)
if HAS_TF:
    tf.random.set_seed(RANDOM_STATE)


def load_primary_tumor(path):
    expr = pd.read_csv(path, index_col=0).T
    expr = expr[expr.index.str.endswith("-01")]
    return expr.astype(np.float32)


# ----------------------------- models (copied from the notebook) -----------------------------
def get_fresh_models():
    return {
        "LogisticRegression": LogisticRegression(max_iter=2000, class_weight="balanced"),
        "LinearSVM": LinearSVC(class_weight="balanced", max_iter=5000, random_state=RANDOM_STATE),
        "RandomForest": RandomForestClassifier(n_estimators=RF_N_ESTIMATORS,
                                               class_weight="balanced_subsample",
                                               random_state=RANDOM_STATE, n_jobs=-1),
        "AdaBoost": AdaBoostClassifier(n_estimators=ADABOOST_N_ESTIMATORS, random_state=RANDOM_STATE),
    }


def to_proba(model, X):
    if hasattr(model, "predict_proba"):
        return model.predict_proba(X)[:, 1]
    s = model.decision_function(X)
    return (s - s.min()) / (s.max() - s.min() + 1e-9)


def run_sklearn(model, Xtr, ytr, Xva, Xte):
    model.fit(Xtr, ytr)
    return to_proba(model, Xva), to_proba(model, Xte)


def run_xgb(Xtr, ytr, Xva, yva, Xte):
    m = xgb.XGBClassifier(n_estimators=200, max_depth=4, learning_rate=0.08,
                          tree_method="hist", max_bin=63, eval_metric="logloss",
                          random_state=RANDOM_STATE, n_jobs=N_CPU_THREADS)
    m.fit(Xtr, ytr, eval_set=[(Xva, yva)], verbose=False)
    return m.predict_proba(Xva)[:, 1], m.predict_proba(Xte)[:, 1]


def run_lgb(Xtr, ytr, Xva, yva, Xte):
    train_ds = lgb.Dataset(Xtr, label=ytr, params={"max_bin": 63, "feature_pre_filter": False})
    val_ds = lgb.Dataset(Xva, label=yva, reference=train_ds)
    p = {"objective": "binary", "metric": "auc", "verbose": -1,
         "num_threads": N_CPU_THREADS, "max_bin": 63,
         "num_leaves": 31, "learning_rate": 0.08, "max_depth": 4}
    bst = lgb.train(p, train_ds, num_boost_round=300, valid_sets=[val_ds],
                    callbacks=[lgb.early_stopping(20, verbose=False)])
    return bst.predict(Xva), bst.predict(Xte)


def build_cnn(n_features):
    model = keras.Sequential([
        keras.layers.Input(shape=(n_features, 1)),
        keras.layers.Conv1D(16, kernel_size=5, activation="relu"),
        keras.layers.Flatten(),
        keras.layers.Dropout(0.4),
        keras.layers.Dense(32, activation="relu"),
        keras.layers.Dropout(0.3),
        keras.layers.Dense(1, activation="sigmoid"),
    ])
    model.compile(optimizer=keras.optimizers.Adam(learning_rate=0.001),
                  loss="binary_crossentropy", metrics=["AUC"])
    return model


def run_cnn(Xtr, ytr, Xva, yva, Xte):
    sc = StandardScaler().fit(Xtr)
    Xtr_s, Xva_s, Xte_s = sc.transform(Xtr), sc.transform(Xva), sc.transform(Xte)
    model = build_cnn(Xtr_s.shape[1])
    es = keras.callbacks.EarlyStopping(monitor="val_loss", patience=8, restore_best_weights=True)
    model.fit(Xtr_s[..., None], ytr, validation_data=(Xva_s[..., None], yva),
              epochs=CNN_EPOCHS, batch_size=16, callbacks=[es], verbose=0)
    return (model.predict(Xva_s[..., None], verbose=0).flatten(),
            model.predict(Xte_s[..., None], verbose=0).flatten())


def metrics(y, p, prefix):
    pred = (p >= 0.5).astype(int)
    return {f"{prefix}_Accuracy": accuracy_score(y, pred),
            f"{prefix}_AUC": roc_auc_score(y, p),
            f"{prefix}_F1": f1_score(y, pred)}


def holm(pvals):
    order = np.argsort(pvals)
    adj = np.empty(len(pvals))
    running = 0.0
    for rank, i in enumerate(order):
        running = max(running, min(1.0, (len(pvals) - rank) * pvals[i]))
        adj[i] = running
    return adj


def main(argv=None):
    ap = argparse.ArgumentParser()
    ap.add_argument("--cohort", choices=["tcga", "gse"], default="tcga")
    ap.add_argument("--data-dir", default="../data")
    ap.add_argument("--out", default=None)
    ap.add_argument("--reference", default=None,
                    help="model_comparison_raw.csv saved by the notebook (found automatically)")
    ap.add_argument("--quick", action="store_true",
                    help="3 splits with small models, only to check that everything runs")
    args = ap.parse_args(argv)
    global N_COMPARISON_REPEATS, CNN_EPOCHS, ADABOOST_N_ESTIMATORS, RF_N_ESTIMATORS
    if args.quick:
        N_COMPARISON_REPEATS, CNN_EPOCHS, ADABOOST_N_ESTIMATORS, RF_N_ESTIMATORS = 3, 5, 20, 100
    else:  # full settings (set explicitly so a quick run in the same session cannot leak into a full run)
        N_COMPARISON_REPEATS, CNN_EPOCHS, ADABOOST_N_ESTIMATORS, RF_N_ESTIMATORS = 20, 100, 200, 500
    # same seeds as the original notebooks, set right before the comparison loop
    np.random.seed(RANDOM_STATE)
    if HAS_TF:
        tf.random.set_seed(RANDOM_STATE)
    args.out = args.out or f"../results/model_selection_{args.cohort}" + ("_quick" if args.quick else "")
    os.makedirs(args.out, exist_ok=True)
    raw_path = os.path.join(args.out, "model_selection_raw.csv")

    if args.cohort == "tcga":
        X_luad = load_primary_tumor(os.path.join(args.data_dir, "LUAD.csv"))
        X_lusc = load_primary_tumor(os.path.join(args.data_dir, "LUSC.csv"))
        genes = X_luad.columns.intersection(X_lusc.columns)
        X_vals = pd.concat([X_luad[genes], X_lusc[genes]], axis=0).values
        y_bin = np.array([0] * len(X_luad) + [1] * len(X_lusc))
    else:  # identical to gse37745_full_pipeline.ipynb
        expr_path = os.path.join(args.data_dir, "GSE37745_expression.csv")
        if not os.path.exists(expr_path):
            expr_path += ".gz"
        expr = pd.read_csv(expr_path, index_col=0).T
        expr = expr.loc[:, ~expr.columns.duplicated()]
        clin = pd.read_csv(os.path.join(args.data_dir, "GSE37745_clinical.csv"), index_col=0)
        clin = clin[clin["histology"].isin(["LUAD", "LUSC"])]
        X_vals = expr.loc[clin.index].astype(np.float32).values
        y_bin = (clin["histology"] == "LUSC").astype(int).values
    print(f"Cohort: {args.cohort.upper()} | Data: {X_vals.shape[0]} samples, {X_vals.shape[1]} genes | TensorFlow: {HAS_TF}")

    rows = pd.read_csv(raw_path).to_dict("records") if os.path.exists(raw_path) else []
    done = {r["seed"] for r in rows}
    t0 = time.time()
    print(f"Running {N_COMPARISON_REPEATS} splits{' (QUICK TEST)' if args.quick else ''} ...", flush=True)
    for seed in range(N_COMPARISON_REPEATS):
        if seed in done:
            continue
        Xtr, Xtemp, ytr, ytemp = train_test_split(X_vals, y_bin, test_size=0.2,
                                                  random_state=seed, stratify=y_bin)
        Xva, Xte, yva, yte = train_test_split(Xtemp, ytemp, test_size=0.5,
                                              random_state=seed, stratify=ytemp)
        results = {}
        for name, model in get_fresh_models().items():
            results[name] = run_sklearn(model, Xtr, ytr, Xva, Xte)
        results["XGBoost"] = run_xgb(Xtr, ytr, Xva, yva, Xte)
        results["LightGBM"] = run_lgb(Xtr, ytr, Xva, yva, Xte)
        if HAS_TF:
            results["CNN (Conv1D)"] = run_cnn(Xtr, ytr, Xva, yva, Xte)
        for name, (pva, pte) in results.items():
            rows.append({"seed": seed, "Model": name, **metrics(yva, pva, "Val"), **metrics(yte, pte, "Test")})
        pd.DataFrame(rows).to_csv(raw_path, index=False)
        print(f"  split {seed + 1}/{N_COMPARISON_REPEATS} done ({time.time() - t0:.0f}s)", flush=True)

    df = pd.DataFrame(rows)
    lines = []

    # 0. reproduction check against the original comparison (test set)
    ref_path = args.reference or os.path.join(
        "../results/final_pipeline" if args.cohort == "tcga" else "../results/gse37745_pipeline",
        "model_comparison_raw.csv")
    lines.append("=== 0. Reproduction check: test results vs the original notebook run ===")
    if os.path.exists(ref_path) and not args.quick:
        ref = pd.read_csv(ref_path)
        merged = df.merge(ref, on=["seed", "Model"], how="inner")
        if len(merged):
            d = (merged["Test_Accuracy"] - merged["Accuracy"]).abs()
            lines.append(f"Compared {len(merged)} model-split results with {ref_path}")
            lines.append(f"Identical test accuracy: {int((d < 1e-9).sum())}/{len(merged)}; "
                         f"largest difference: {d.max():.4f}")
            per_model = merged.assign(diff=d).groupby("Model")["diff"].max().round(4).to_dict()
            lines.append("Largest difference per model: " + repr(per_model))
        else:
            lines.append("No matching rows found in the reference file.")
    else:
        lines.append("Skipped (quick mode or reference file not found: " + ref_path + ")")
    lines.append("")

    # 1. selection by mean validation accuracy
    summ = (df.groupby("Model")
              .agg(Val_Acc=("Val_Accuracy", "mean"), Val_AUC=("Val_AUC", "mean"),
                   Test_Acc=("Test_Accuracy", "mean"), Test_Acc_SD=("Test_Accuracy", "std"),
                   Test_AUC=("Test_AUC", "mean"), Test_AUC_SD=("Test_AUC", "std"))
              .sort_values(["Val_Acc", "Val_AUC"], ascending=False))
    summ.to_csv(os.path.join(args.out, "model_selection_summary.csv"))
    lines.append(f"=== Mean over {df.seed.nunique()} splits, sorted by VALIDATION accuracy ===")
    lines.append(summ.round(4).to_string())
    lines.append(f"\nSelected by validation accuracy: {summ.index[0]}")
    lines.append("(Note: XGBoost, LightGBM and the CNN also used the validation set for early stopping.)")

    # 2. per-split selection by validation accuracy, scored on test
    picks = []
    for seed, g in df.groupby("seed"):
        best = g.sort_values(["Val_Accuracy", "Val_AUC"], ascending=False).iloc[0]
        picks.append({"seed": seed, "picked": best["Model"],
                      "Test_Accuracy": best["Test_Accuracy"], "Test_AUC": best["Test_AUC"]})
    picks = pd.DataFrame(picks)
    picks.to_csv(os.path.join(args.out, "per_split_selection.csv"), index=False)
    lines.append("\n=== Per-split selection (best validation accuracy in each split, scored on its test set) ===")
    lines.append(f"Test accuracy: {picks.Test_Accuracy.mean() * 100:.2f} +- {picks.Test_Accuracy.std() * 100:.2f}%  "
                 f"| Test AUC: {picks.Test_AUC.mean():.4f} +- {picks.Test_AUC.std():.4f}")
    lines.append("Times each model was picked: " + picks.picked.value_counts().to_dict().__repr__())

    # 3. statistical comparison of classifiers on the paired test results
    models = list(summ.index)
    for metric in ["Test_Accuracy", "Test_AUC"]:
        wide = df.pivot(index="seed", columns="Model", values=metric)[models]
        stat, p = friedmanchisquare(*[wide[m].values for m in models])
        lines.append(f"\n=== {metric}: Friedman test over {len(wide)} paired splits ===")
        lines.append(f"chi2 = {stat:.3f}, p = {p:.4g}")
        lines.append("Mean rank (1 = best): " +
                     wide.rank(axis=1, ascending=False).mean().sort_values().round(2).to_dict().__repr__())
        # paired tests against logistic regression and, if different, the validation-selected model
        for ref_model in dict.fromkeys(["LogisticRegression", summ.index[0]]):
            others = [m for m in models if m != ref_model]
            raw_p = []
            for m in others:
                d = wide[ref_model] - wide[m]
                raw_p.append(wilcoxon(wide[ref_model], wide[m]).pvalue if np.any(d != 0) else 1.0)
            adj = holm(np.array(raw_p))
            lines.append(f"Wilcoxon signed-rank, {ref_model} vs each model (Holm-adjusted p):")
            for m, pr, pa in zip(others, raw_p, adj):
                diff = (wide[ref_model] - wide[m]).mean()
                lines.append(f"  vs {m:<14} mean diff {diff:+.4f}  p = {pr:.4g}  Holm p = {pa:.4g}")

    report = "\n".join(lines)
    print("\n" + report)
    with open(os.path.join(args.out, "model_selection_report.txt"), "w", encoding="utf-8") as f:
        f.write(report + "\n")
    print(f"\nSaved to {args.out}/")

## 2. Quick test 


In [2]:
main(["--cohort", "tcga", "--data-dir", "../data", "--quick"])

Cohort: TCGA | Data: 1017 samples, 20530 genes | TensorFlow: True
Running 3 splits (QUICK TEST) ...
  split 1/3 done (422s)
  split 2/3 done (797s)
  split 3/3 done (1184s)

=== 0. Reproduction check: test results vs the original notebook run ===
Skipped (quick mode or reference file not found: results_final_pipeline\model_comparison_raw.csv)

=== Mean over 3 splits, sorted by VALIDATION accuracy ===
                    Val_Acc  Val_AUC  Test_Acc  Test_Acc_SD  Test_AUC  Test_AUC_SD
Model                                                                             
LightGBM             0.9412   0.9894    0.9706       0.0259    0.9955       0.0046
XGBoost              0.9412   0.9872    0.9706       0.0259    0.9965       0.0033
CNN (Conv1D)         0.9412   0.9853    0.9575       0.0226    0.9936       0.0069
LogisticRegression   0.9346   0.9869    0.9804       0.0259    0.9953       0.0055
LinearSVM            0.9346   0.9865    0.9706       0.0294    0.9954       0.0043
RandomForest   

## 3. TCGA full run 
Results: `results_model_selection_tcga/`

In [3]:
main(["--cohort", "tcga", "--data-dir", "../data"])

Cohort: TCGA | Data: 1017 samples, 20530 genes | TensorFlow: True
Running 20 splits ...
  split 1/20 done (1433s)
  split 2/20 done (2754s)
  split 3/20 done (3965s)
  split 4/20 done (4664s)
  split 5/20 done (9390s)
  split 6/20 done (10303s)
  split 7/20 done (58902s)
  split 8/20 done (60292s)
  split 9/20 done (61207s)
  split 10/20 done (62148s)
  split 11/20 done (63035s)
  split 12/20 done (64178s)
  split 13/20 done (80468s)
  split 14/20 done (81832s)
  split 15/20 done (82998s)
  split 16/20 done (83972s)
  split 17/20 done (84950s)
  split 18/20 done (86552s)
  split 19/20 done (88331s)
  split 20/20 done (89995s)

=== 0. Reproduction check: test results vs the original notebook run ===
Compared 140 model-split results with results_final_pipeline\model_comparison_raw.csv
Identical test accuracy: 124/140; largest difference: 0.0490
Largest difference per model: {'AdaBoost': 0.0, 'CNN (Conv1D)': 0.049, 'LightGBM': 0.0, 'LinearSVM': 0.0, 'LogisticRegression': 0.0, 'RandomFores

## 4. GSE37745 full run 
Results: `results_model_selection_gse/`

In [4]:
main(["--cohort", "gse", "--data-dir", "../data"])

Cohort: GSE | Data: 172 samples, 21655 genes | TensorFlow: True
Running 20 splits ...
  split 1/20 done (389s)
  split 2/20 done (761s)
  split 3/20 done (983s)
  split 4/20 done (1180s)
  split 5/20 done (1381s)
  split 6/20 done (1590s)
  split 7/20 done (1857s)
  split 8/20 done (2078s)
  split 9/20 done (2360s)
  split 10/20 done (2619s)
  split 11/20 done (3008s)
  split 12/20 done (3230s)
  split 13/20 done (3474s)
  split 14/20 done (4121s)
  split 15/20 done (4320s)
  split 16/20 done (4534s)
  split 17/20 done (4730s)
  split 18/20 done (4983s)
  split 19/20 done (7825s)
  split 20/20 done (8018s)

=== 0. Reproduction check: test results vs the original notebook run ===
Compared 140 model-split results with results_gse37745_pipeline\model_comparison_raw.csv
Identical test accuracy: 134/140; largest difference: 0.2778
Largest difference per model: {'AdaBoost': 0.0, 'CNN (Conv1D)': 0.2778, 'LightGBM': 0.0, 'LinearSVM': 0.0, 'LogisticRegression': 0.0, 'RandomForest': 0.0, 'XGBoos